<a href="https://colab.research.google.com/github/rbnascimentoo/desafio-pratico-analise-financeira-com-python/blob/main/Desafio_pr%C3%A1tico_An%C3%A1lise_Financeira_com_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Desafio prático: Análise Financeira com Python

In [52]:
import os
import csv
import random
from datetime import datetime, timedelta

In [53]:
def gerar_dados(nome_arquivo='transacoes.csv'):
  if os.path.exists(nome_arquivo):
    print(f"Arquivo '{nome_arquivo}' já existe.")
    return

  print("Gerando arquivo...")

  transacoes = []

  data_inicial = datetime(2026, 1, 1)

  categorias_opcoes = ["salario", "compra", "transferencia", "investimento", "conta", "alimentacao"]

  # descrição dinâmica
  descricoes = {
      "salario": ["Salário mensal", "Adiantamento", "Bônus"],
      "compra": ["Supermercado", "Eletrônicos", "Roupas"],
      "transferencia": ["Pix recebido", "Transferência enviada"],
      "investimento": ["Compra cota VISC11", "Rendimento HGLG11", "Venda KNCR11", "Compra XPML11", "Compra cota CPTS11"],
      "conta": ["Conta de Luz", "Internet", "Condomínio"],
      "alimentacao": ["Restaurante", "iFood", "Padaria"]
  }

  for i in range(26):
    data_transacao = data_inicial + timedelta(days=random.randint(0, 100)) # 3 meses
    categoria = random.choice(categorias_opcoes)
    tipo = "credito" if categoria in ["salario"] else random.choice(["credito", "debito"])
    descricao = random.choice(descricoes[categoria])

    if categoria == "salario":
        valor = round(random.uniform(3500, 8000), 2)
    else:
        valor = round(random.uniform(20, 1500), 2)

    transacoes.append({
          "data": data_transacao.strftime("%Y-%m-%d"),
          "cliente_id": f"CLI{random.randint(1, 20):03d}",
          "tipo": tipo,
          "valor": valor,
          "descricao": descricao,
          "categoria": categoria
    })

  # Gera itens com acima de R$10000
  transacoes.extend([
        {"data": "2026-02-15", "cliente_id": "CLI001", "tipo": "credito", "valor": 15000.00, "descricao": "Venda de veículo", "categoria": "transferencia"},
        {"data": "2026-03-20", "cliente_id": "CLI002", "tipo": "debito", "valor": 12500.00, "descricao": "Pagamento de reforma", "categoria": "servicos"},
        {"data": "2026-03-10", "cliente_id": "CLI015", "tipo": "debito", "valor": 35000.00, "descricao": "Aporte maciço KNCR11", "categoria": "investimento"},
        {"data": "2026-04-02", "cliente_id": "CLI005", "tipo": "credito", "valor": 18500.00, "descricao": "Liquidação FII HGLG11", "categoria": "investimento"}
  ])
  # Gera itens inválidos
  transacoes_invalidas = [
      {"data": "2026-01-10", "cliente_id": "CLI021", "tipo": "debito", "valor": -50.00, "descricao": "Erro valor negativo", "categoria": "erro"},
      {"data": "2026-02-28", "cliente_id": "", "tipo": "credito", "valor": 500.00, "descricao": "Erro sem cliente", "categoria": "erro"},
      {"data": "20-03-2026", "cliente_id": "CLI022", "tipo": "debito", "valor": 100.00, "descricao": "Erro formato data", "categoria": "erro"},
      {"data": "2026-04-05", "cliente_id": "CLI023", "tipo": "pix", "valor": 200.00, "descricao": "Erro tipo transação", "categoria": "erro"},
      {"data": "2026-04-10", "cliente_id": "CLI024", "tipo": "debito", "valor": "abc", "descricao": "Erro valor string", "categoria": "erro"},
      {"data": "2026-01-25", "cliente_id": "CLI025", "tipo": "debito", "valor": 0.00, "descricao": "Erro valor zero", "categoria": "erro"},
      {"data": "2026-02-10", "cliente_id": "CLI026", "tipo": "TED", "valor": 150.00, "descricao": "Erro tipo transacao 2", "categoria": "erro"},
      {"data": "25/02/2026", "cliente_id": "CLI027", "tipo": "credito", "valor": 300.00, "descricao": "Erro data padrao BR", "categoria": "erro"},
      {"id_corrompido": "ABC", "data": "2026-03-05", "cliente_id": "CLI028", "tipo": "debito", "valor": 80.00, "descricao": "Erro ID string", "categoria": "erro"},
      {"id_corrompido": "", "data": "2026-03-12", "cliente_id": "CLI029", "tipo": "credito", "valor": 120.00, "descricao": "Erro ID vazio", "categoria": "erro"}
  ]
  transacoes.extend(transacoes_invalidas)

  random.shuffle(transacoes) # Mistura válidos e inválidos

  with open(nome_arquivo, mode='w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=["id", "data", "cliente_id", "tipo", "valor", "descricao", "categoria"])
    writer.writeheader()

    for index, t in enumerate(transacoes, start=1):
        if "id_corrompido" in t:
            t["id"] = t.pop("id_corrompido")
        else:
            t["id"] = index
        writer.writerow(t)

  print(f"✓ Arquivo '{nome_arquivo}' criado com {len(transacoes)} registros (embaralhados).")

In [54]:
gerar_dados()

Arquivo 'transacoes.csv' já existe.


In [55]:
def validar_transacao(linha):
  try:
    id_transacao = int(linha.get('id', ''))
  except ValueError:
    raise ValueError("ID deve ser um número inteiro.")

  cliente_id = linha.get('cliente_id', '').strip()
  if not cliente_id:
      raise ValueError("Cliente ID vazio")

  try:
      data_obj = datetime.strptime(linha.get('data', '').strip(), '%Y-%m-%d')
  except ValueError:
      raise ValueError("Formato de data inválido")

  tipo = linha.get('tipo', '').strip().lower()
  if tipo not in ['credito', 'debito']:
      raise ValueError("Tipo de transação desconhecido")

  try:
      valor = float(linha.get('valor', ''))
      if valor <= 0:
          raise ValueError("Valor negativo ou zero")
  except ValueError:
      raise ValueError("Valor não numérico")

  return {
        'id': id_transacao,
        'data_obj': data_obj,
        'data_str': data_obj.strftime('%Y-%m'),
        'cliente_id': cliente_id,
        'tipo': tipo,
        'valor': valor,
        'categoria': linha.get('categoria', '')
    }


In [56]:
def ler_transacoes(nome_arquivo='transacoes.csv'):
  transacoes_limpas = []
  total_lidas = 0
  total_invalidas = 0

  try:
    with open(nome_arquivo, mode='r', encoding='utf-8') as f:
      leitor = csv.DictReader(f)
      for linha in leitor:
        total_lidas += 1
        try:
          transacao_valida = validar_transacao(linha)
          transacoes_limpas.append(transacao_valida)
        except ValueError:
          total_invalidas += 1
  except FileNotFoundError:
    print(f"Erro: O arquivo '{nome_arquivo}' não foi encontrado no sistema.")
    return [], 0, 0

  print("--- Resumo da Leitura e Limpeza ---")
  print(f"Total de linhas lidas: {total_lidas}")
  print(f"Linhas válidas aprovadas: {len(transacoes_limpas)}")
  print(f"Linhas inválidas descartadas: {total_invalidas}\n")

  return transacoes_limpas, len(transacoes_limpas), total_invalidas

In [57]:
ler_transacoes()

--- Resumo da Leitura e Limpeza ---
Total de linhas lidas: 40
Linhas válidas aprovadas: 30
Linhas inválidas descartadas: 10



([{'id': 1,
   'data_obj': datetime.datetime(2026, 2, 17, 0, 0),
   'data_str': '2026-02',
   'cliente_id': 'CLI019',
   'tipo': 'debito',
   'valor': 1408.96,
   'categoria': 'compra'},
  {'id': 2,
   'data_obj': datetime.datetime(2026, 1, 2, 0, 0),
   'data_str': '2026-01',
   'cliente_id': 'CLI007',
   'tipo': 'credito',
   'valor': 189.6,
   'categoria': 'investimento'},
  {'id': 3,
   'data_obj': datetime.datetime(2026, 1, 18, 0, 0),
   'data_str': '2026-01',
   'cliente_id': 'CLI009',
   'tipo': 'credito',
   'valor': 693.96,
   'categoria': 'conta'},
  {'id': 4,
   'data_obj': datetime.datetime(2026, 1, 26, 0, 0),
   'data_str': '2026-01',
   'cliente_id': 'CLI020',
   'tipo': 'credito',
   'valor': 985.21,
   'categoria': 'investimento'},
  {'id': 7,
   'data_obj': datetime.datetime(2026, 4, 7, 0, 0),
   'data_str': '2026-04',
   'cliente_id': 'CLI013',
   'tipo': 'credito',
   'valor': 73.46,
   'categoria': 'transferencia'},
  {'id': 10,
   'data_obj': datetime.datetime(2026,

In [60]:
lista_final, validas, invalidas = ler_transacoes()

if lista_final:
    print("Primeiro registro válido aprovado:")
    print(lista_final[0])

--- Resumo da Leitura e Limpeza ---
Total de linhas lidas: 40
Linhas válidas aprovadas: 30
Linhas inválidas descartadas: 10

Primeiro registro válido aprovado:
{'id': 1, 'data_obj': datetime.datetime(2026, 2, 17, 0, 0), 'data_str': '2026-02', 'cliente_id': 'CLI019', 'tipo': 'debito', 'valor': 1408.96, 'categoria': 'compra'}


In [67]:
def gerar_relatorio(transacoes, total_validadas, total_invalidas):
  LIMITE_SUSPEITO = 10000.00

  if not transacoes:
    print("Nenhuma transação para gerar relatório.")
    return None

  relatorio = {
    "gerado_em": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "total_transacoes_validas": total_validadas, # Corrected variable name
    "total_transacoes_invalidas": total_invalidas,
    "periodo_analisado": {},
    "resumo_mensal": {},
    "transacoes_suspeitas": []
  }

  datas_objetos = []

  for t in transacoes:
        mes_ano = t['data_str']
        valor = t['valor']
        datas_objetos.append(t['data_obj'])

        if valor > LIMITE_SUSPEITO:
            relatorio["transacoes_suspeitas"].append({
                "id": t['id'],
                "cliente_id": t['cliente_id'],
                "data": t['data_obj'].strftime('%Y-%m-%d'),
                "valor": valor,
                "categoria": t['categoria']
            })

        if mes_ano not in relatorio["resumo_mensal"]:
            relatorio["resumo_mensal"][mes_ano] = {
                "quantidade_transacoes": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "_valores_brutos": []
            }

        mes_ref = relatorio["resumo_mensal"][mes_ano]
        mes_ref["quantidade_transacoes"] += 1
        mes_ref["_valores_brutos"].append(valor)

        if t['tipo'] == 'credito':
            mes_ref["total_credito"] += valor
        elif t['tipo'] == 'debito':
            mes_ref["total_debito"] += valor

  for mes, dados in relatorio["resumo_mensal"].items():
      dados["saldo_mensal"] = round(dados["total_credito"] - dados["total_debito"], 2)

      lista_valores = dados["_valores_brutos"]
      dados["media_transacoes"] = round(sum(lista_valores) / len(lista_valores), 2)
      dados["maior_transacao"] = max(lista_valores)
      dados["menor_transacao"] = min(lista_valores)

      del dados["_valores_brutos"]

  data_inicio = min(datas_objetos)
  data_fim = max(datas_objetos)
  dias_corridos = (data_fim - data_inicio).days

  relatorio["periodo_analisado"] = {
      "data_inicio": data_inicio.strftime('%Y-%m-%d'),
      "data_fim": data_fim.strftime('%Y-%m-%d'),
      "dias_corridos": dias_corridos
  }

  return relatorio

In [68]:
lista_final, validas, invalidas = ler_transacoes()

gerar_relatorio(lista_final, validas, invalidas)

--- Resumo da Leitura e Limpeza ---
Total de linhas lidas: 40
Linhas válidas aprovadas: 30
Linhas inválidas descartadas: 10



{'gerado_em': '2026-06-16 01:27:34',
 'total_transacoes_validas': 30,
 'total_transacoes_invalidas': 10,
 'periodo_analisado': {'data_inicio': '2026-01-02',
  'data_fim': '2026-04-07',
  'dias_corridos': 95},
 'resumo_mensal': {'2026-02': {'quantidade_transacoes': 8,
   'total_credito': 24840.64,
   'total_debito': 2049.21,
   'saldo_mensal': 22791.43,
   'media_transacoes': 3361.23,
   'maior_transacao': 15000.0,
   'menor_transacao': 20.63},
  '2026-01': {'quantidade_transacoes': 9,
   'total_credito': 14385.07,
   'total_debito': 3119.1000000000004,
   'saldo_mensal': 11265.97,
   'media_transacoes': 1944.91,
   'maior_transacao': 6078.37,
   'menor_transacao': 189.6},
  '2026-04': {'quantidade_transacoes': 2,
   'total_credito': 18573.46,
   'total_debito': 0.0,
   'saldo_mensal': 18573.46,
   'media_transacoes': 9286.73,
   'maior_transacao': 18500.0,
   'menor_transacao': 73.46},
  '2026-03': {'quantidade_transacoes': 11,
   'total_credito': 7980.1,
   'total_debito': 51281.83,
 